# 06 — Combat & Damage Analysis: Who Kills What, and How?

In Overwatch, understanding *how* kills happen is just as important as understanding *who* gets them. This notebook digs into the combat layer of the Parsertime dataset to answer:

- **Which abilities are the most lethal?** (Primary fire vs cooldowns vs ultimates)
- **Do assists predict wins?** (Offensive vs defensive assists and their correlation with match outcomes)
- **Which heroes convert eliminations into final blows most efficiently?**
- **When do kills happen?** (Early-match vs late-match kill patterns)
- **How important are critical hits?**

### Why This Matters for Coaches
Understanding lethal ability usage helps coaches identify which cooldowns and ultimates their team should be tracking. Assist patterns reveal whether your team's value comes from enabling kills (offensive assists) or preventing them (defensive assists). Kill timing patterns show whether your team is strong in the opener or the closer.

### Data Sources
- `Kill` table: ~373K individual kill events with attacker, victim, ability, critical hit, and timing info
- `PlayerStat` table: per-player per-round aggregated stats (eliminations, final blows, assists)
- `OffensiveAssist` / `DefensiveAssist` tables: individual assist events
- `MatchStart` / `MatchEnd` tables: match outcomes for win correlation

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_csv, load_kills, load_player_stats, load_matches
from src.preprocessing import (
    determine_match_winner, add_role_column, HERO_ROLES,
    enrich_kills_with_match_info
)
from src.metrics import final_blow_ratio
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig, role_color

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load and Prepare Data

In [ ]:
kills = load_kills()
player_stats = load_player_stats()
match_start, match_end = load_matches()
matches = determine_match_winner(match_end, match_start)
off_assists = load_csv('OffensiveAssist')
def_assists = load_csv('DefensiveAssist')

# Enrich kills with match info (map, winner)
kills_enriched = enrich_kills_with_match_info(kills, matches)

# Add role columns
kills_enriched = add_role_column(kills_enriched, hero_col='attacker_hero')
kills_enriched = kills_enriched.rename(columns={'role': 'attacker_role'})
kills_enriched['victim_role'] = kills_enriched['victim_hero'].map(HERO_ROLES).fillna('Unknown').astype('category')

# Filter to inter-team kills (exclude self/environmental)
combat_kills = kills_enriched[
    (kills_enriched['attacker_team'] != kills_enriched['victim_team']) &
    (kills_enriched['attacker_name'] != kills_enriched['victim_name'])
].copy()

print(f"Total kills: {len(kills):,}")
print(f"Inter-team combat kills: {len(combat_kills):,}")
print(f"Matches with outcomes: {len(matches):,}")
print(f"Offensive assists: {len(off_assists):,}")
print(f"Defensive assists: {len(def_assists):,}")

---
## 2. Most Lethal Abilities

In Overwatch, every kill is tagged with the ability that dealt the final blow. This lets us see whether kills come from primary fire (basic attacks), cooldown abilities (like Ana's Sleep Dart), or ultimates (like Genji's Dragonblade).

> **Coaching insight**: If a particular ability secures a disproportionate number of kills, it tells you that ability is a high-value resource. Tracking its cooldown in team comms becomes critical.

In [ ]:
# Top 25 most lethal abilities across all kills
ability_kills = combat_kills['event_ability'].value_counts().head(25)

fig, ax = plt.subplots(figsize=(14, 9))
bars = ax.barh(ability_kills.index[::-1], ability_kills.values[::-1],
               color=OW_COLORS['orange'], alpha=0.9, edgecolor=OW_COLORS['dark_blue'])

# Add count labels
for bar, val in zip(bars, ability_kills.values[::-1]):
    ax.text(val + ability_kills.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color=OW_COLORS['white'])

ax.set_xlabel('Total Kills')
ax.set_title('Top 25 Most Lethal Abilities Across All Matches', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, '06_top_abilities')
plt.show()

# Show percentage breakdown
total = len(combat_kills)
print(f"\nTop 10 abilities account for {ability_kills.head(10).sum()/total*100:.1f}% of all kills")
print(f"Primary Fire alone accounts for {ability_kills.get('Primary Fire', 0)/total*100:.1f}% of kills")

In [ ]:
# Most lethal abilities per role
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, role_name in zip(axes, ['Tank', 'DPS', 'Support']):
    role_kills = combat_kills[combat_kills['attacker_role'] == role_name]
    top_abilities = role_kills['event_ability'].value_counts().head(12)
    
    ax.barh(top_abilities.index[::-1], top_abilities.values[::-1],
            color=ROLE_COLORS[role_name], alpha=0.85)
    ax.set_xlabel('Kills')
    ax.set_title(f'{role_name} — Top Abilities', fontsize=13)

plt.suptitle('Most Lethal Abilities by Role', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '06_abilities_by_role')
plt.show()

### Ability Kill Summary Table

In [ ]:
# Summary table: ability + hero combinations
hero_ability = combat_kills.groupby(['attacker_hero', 'event_ability']).size().reset_index(name='kills')
hero_ability['role'] = hero_ability['attacker_hero'].map(HERO_ROLES)
hero_ability = hero_ability.sort_values('kills', ascending=False)

print("Top 20 Hero + Ability Kill Combinations:")
print("=" * 55)
display_df = hero_ability.head(20)[['attacker_hero', 'event_ability', 'role', 'kills']].reset_index(drop=True)
display_df.index += 1
display_df.columns = ['Hero', 'Ability', 'Role', 'Kills']
display_df

---
## 3. Offensive vs Defensive Assists and Win Correlation

Overwatch tracks two types of assists:
- **Offensive assists**: Enabling a kill (e.g., Discord Orb, damage boost, speed boost into a kill)
- **Defensive assists**: Preventing a death (e.g., healing someone who was about to die, using a barrier)

We want to know: **Do teams with more assists win more?** And is offensive or defensive assistance more predictive?

> **Coaching insight**: If offensive assists correlate more strongly with wins, your team needs to focus on *enabling* kills. If defensive assists matter more, survivability and peel are the priority.

In [ ]:
# Count assists per team per match
off_per_match = off_assists.groupby(['MapDataId', 'player_team']).size().reset_index(name='offensive_assists')
def_per_match = def_assists.groupby(['MapDataId', 'player_team']).size().reset_index(name='defensive_assists')

# Merge together
assists_combined = off_per_match.merge(def_per_match, on=['MapDataId', 'player_team'], how='outer').fillna(0)

# Join with match outcomes to determine if this team won
# Build a team-level match outcome table
team_outcomes = []
for _, m in matches.iterrows():
    team_outcomes.append({'MapDataId': m['MapDataId'], 'player_team': m['team_1_name'],
                          'won': m['winner'] == m['team_1_name']})
    team_outcomes.append({'MapDataId': m['MapDataId'], 'player_team': m['team_2_name'],
                          'won': m['winner'] == m['team_2_name']})
team_outcomes = pd.DataFrame(team_outcomes)

assists_with_outcome = assists_combined.merge(team_outcomes, on=['MapDataId', 'player_team'], how='inner')

print(f"Team-match records with assists and outcomes: {len(assists_with_outcome):,}")
print(f"Win rate in dataset: {assists_with_outcome['won'].mean()*100:.1f}%")

In [ ]:
# Compare assist counts: winners vs losers
winners = assists_with_outcome[assists_with_outcome['won'] == True]
losers = assists_with_outcome[assists_with_outcome['won'] == False]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Offensive assists
axes[0].hist(winners['offensive_assists'], bins=40, alpha=0.7, color=OW_COLORS['green'],
             label=f'Winners (mean={winners["offensive_assists"].mean():.0f})', density=True)
axes[0].hist(losers['offensive_assists'], bins=40, alpha=0.7, color=OW_COLORS['red'],
             label=f'Losers (mean={losers["offensive_assists"].mean():.0f})', density=True)
axes[0].set_xlabel('Offensive Assists per Match')
axes[0].set_ylabel('Density')
axes[0].set_title('Offensive Assists: Winners vs Losers')
axes[0].legend()

# Defensive assists
axes[1].hist(winners['defensive_assists'], bins=40, alpha=0.7, color=OW_COLORS['green'],
             label=f'Winners (mean={winners["defensive_assists"].mean():.0f})', density=True)
axes[1].hist(losers['defensive_assists'], bins=40, alpha=0.7, color=OW_COLORS['red'],
             label=f'Losers (mean={losers["defensive_assists"].mean():.0f})', density=True)
axes[1].set_xlabel('Defensive Assists per Match')
axes[1].set_ylabel('Density')
axes[1].set_title('Defensive Assists: Winners vs Losers')
axes[1].legend()

plt.tight_layout()
save_fig(fig, '06_assists_winners_vs_losers')
plt.show()

# Statistical comparison
from scipy import stats
for col, label in [('offensive_assists', 'Offensive'), ('defensive_assists', 'Defensive')]:
    t_stat, p_val = stats.ttest_ind(winners[col], losers[col])
    diff = winners[col].mean() - losers[col].mean()
    print(f"{label} assists — Winners avg: {winners[col].mean():.1f}, Losers avg: {losers[col].mean():.1f}, "
          f"Diff: {diff:+.1f}, p={p_val:.2e}")

In [ ]:
# Which heroes generate the most assists?
off_by_hero = off_assists.groupby('player_hero').size().sort_values(ascending=False).head(15)
def_by_hero = def_assists.groupby('player_hero').size().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) for h in off_by_hero.index]
axes[0].barh(off_by_hero.index[::-1], off_by_hero.values[::-1], color=colors[::-1])
axes[0].set_xlabel('Total Offensive Assists')
axes[0].set_title('Top Heroes: Offensive Assists (Kill Enabling)')

colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) for h in def_by_hero.index]
axes[1].barh(def_by_hero.index[::-1], def_by_hero.values[::-1], color=colors[::-1])
axes[1].set_xlabel('Total Defensive Assists')
axes[1].set_title('Top Heroes: Defensive Assists (Death Prevention)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
axes[0].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '06_assists_by_hero')
plt.show()

---
## 4. Final Blow Efficiency by Hero

The **final blow ratio** (final blows / eliminations) measures how often a hero *finishes* kills rather than merely participating. A high ratio means the hero is a closer — they secure kills. A low ratio means they contribute chip damage but rely on teammates to finish.

- **DPS heroes** should have higher ratios (they're the finishers)
- **Support heroes** typically have lower ratios (they contribute damage but aren't expected to close)
- **Tanks** vary: some like Roadhog are executioners, others like Reinhardt are brawlers

> **Coaching insight**: If your DPS players have unusually low final blow ratios, they may be dealing good damage but not finishing targets — a common issue in amateur play.

In [ ]:
# Aggregate PlayerStat per hero (across all players and matches)
hero_stats = player_stats.groupby('player_hero').agg(
    total_elims=('eliminations', 'sum'),
    total_fb=('final_blows', 'sum'),
    total_deaths=('deaths', 'sum'),
    total_time=('hero_time_played', 'sum'),
    records=('id', 'count')
).reset_index()

hero_stats['fb_ratio'] = final_blow_ratio(hero_stats['total_fb'], hero_stats['total_elims'])
hero_stats['role'] = hero_stats['player_hero'].map(HERO_ROLES).fillna('Unknown')

# Filter to heroes with meaningful sample size
hero_stats_filtered = hero_stats[hero_stats['records'] >= 50].sort_values('fb_ratio', ascending=True)

fig, ax = plt.subplots(figsize=(14, 10))
colors = [ROLE_COLORS.get(r, OW_COLORS['light_gray']) for r in hero_stats_filtered['role']]
bars = ax.barh(hero_stats_filtered['player_hero'], hero_stats_filtered['fb_ratio'],
               color=colors, alpha=0.9)

# Add value labels
for bar, (_, row) in zip(bars, hero_stats_filtered.iterrows()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{row["fb_ratio"]:.2f}', va='center', fontsize=8, color=OW_COLORS['white'])

ax.axvline(hero_stats_filtered['fb_ratio'].median(), color=OW_COLORS['gold'], linestyle='--',
           label=f'Median: {hero_stats_filtered["fb_ratio"].median():.2f}')
ax.set_xlabel('Final Blow Ratio (Final Blows / Eliminations)')
ax.set_title('Final Blow Efficiency by Hero (min 50 records)', fontsize=14, fontweight='bold')

legend_elements = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
legend_elements.append(plt.Line2D([0], [0], color=OW_COLORS['gold'], linestyle='--', label='Median'))
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '06_final_blow_ratio')
plt.show()

In [ ]:
# Final blow ratio by role — box plot
player_level = player_stats.copy()
player_level['role'] = player_level['player_hero'].map(HERO_ROLES).fillna('Unknown')
player_level['fb_ratio'] = final_blow_ratio(player_level['final_blows'], player_level['eliminations'])
player_level = player_level[player_level['eliminations'] >= 3]  # minimum to avoid noisy ratios

fig, ax = plt.subplots(figsize=(10, 6))
role_order = ['Tank', 'DPS', 'Support']
bp = ax.boxplot(
    [player_level[player_level['role'] == r]['fb_ratio'].dropna() for r in role_order],
    labels=role_order, patch_artist=True, showfliers=False, widths=0.5
)

for patch, role in zip(bp['boxes'], role_order):
    patch.set_facecolor(ROLE_COLORS[role])
    patch.set_alpha(0.8)
for element in ['whiskers', 'caps', 'medians']:
    for line in bp[element]:
        line.set_color(OW_COLORS['white'])

ax.set_ylabel('Final Blow Ratio')
ax.set_title('Final Blow Ratio Distribution by Role', fontsize=14, fontweight='bold')

# Add means
for i, role in enumerate(role_order):
    mean_val = player_level[player_level['role'] == role]['fb_ratio'].mean()
    ax.scatter(i + 1, mean_val, color=OW_COLORS['gold'], zorder=5, s=80, marker='D',
              label='Mean' if i == 0 else '')

ax.legend()
plt.tight_layout()
save_fig(fig, '06_fb_ratio_by_role')
plt.show()

---
## 5. Kill Timeline Patterns Within Matches

When do kills happen? Overwatch matches have distinct phases:
- **Setup/engage phase** (0-30s): Teams rotate and look for an opening
- **First fight** (30-60s): The initial teamfight
- **Mid-match** (60-300s): Subsequent fights, ult cycling, objective progress
- **Late match / overtime** (300s+): Desperation plays, stagger kills, C9s

> **Coaching insight**: If your team's kill rate drops off in later fights, you may have an ult economy problem. If kills spike early, your team may be good at first fights but struggle in sustain.

In [ ]:
# Kill density over match time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Overall kill density histogram
match_times = combat_kills['match_time'].clip(upper=900)  # Cap at 15 min for viz
axes[0].hist(match_times, bins=90, color=OW_COLORS['orange'], edgecolor=OW_COLORS['dark_blue'],
             alpha=0.9)
axes[0].set_xlabel('Match Time (seconds)')
axes[0].set_ylabel('Number of Kills')
axes[0].set_title('Kill Density Over Match Time', fontsize=14, fontweight='bold')

# Add phase annotations
phases = [(0, 30, 'Setup'), (30, 90, 'First Fight'), (90, 300, 'Mid-Match'), (300, 900, 'Late/OT')]
phase_colors = [OW_COLORS['blue'], OW_COLORS['green'], OW_COLORS['orange'], OW_COLORS['red']]
for (start, end, label), color in zip(phases, phase_colors):
    axes[0].axvspan(start, end, alpha=0.1, color=color)
    axes[0].text((start + end) / 2, axes[0].get_ylim()[1] * 0.9, label,
                 ha='center', fontsize=10, color=color, fontweight='bold')

# Kill rate by role over time (smoothed)
combat_kills_clipped = combat_kills[combat_kills['match_time'] <= 900].copy()
combat_kills_clipped['time_bin'] = (combat_kills_clipped['match_time'] // 30) * 30  # 30s bins

for role_name in ['Tank', 'DPS', 'Support']:
    role_data = combat_kills_clipped[combat_kills_clipped['attacker_role'] == role_name]
    role_timeline = role_data.groupby('time_bin').size()
    axes[1].plot(role_timeline.index, role_timeline.values, color=ROLE_COLORS[role_name],
                 label=role_name, linewidth=2)

axes[1].set_xlabel('Match Time (seconds, 30s bins)')
axes[1].set_ylabel('Kills per 30s Window')
axes[1].set_title('Kill Rate by Role Over Match Time', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
save_fig(fig, '06_kill_timeline')
plt.show()

In [ ]:
# Early vs late kill comparison: do some heroes peak at different times?
combat_kills_clipped = combat_kills[combat_kills['match_time'] <= 600].copy()
combat_kills_clipped['phase'] = pd.cut(
    combat_kills_clipped['match_time'],
    bins=[0, 120, 300, 600],
    labels=['Early (0-2min)', 'Mid (2-5min)', 'Late (5-10min)']
)

# Top 10 heroes in each phase
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, phase_label in zip(axes, ['Early (0-2min)', 'Mid (2-5min)', 'Late (5-10min)']):
    phase_kills = combat_kills_clipped[combat_kills_clipped['phase'] == phase_label]
    top_heroes = phase_kills['attacker_hero'].value_counts().head(10)
    colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) for h in top_heroes.index]
    ax.barh(top_heroes.index[::-1], top_heroes.values[::-1], color=colors[::-1])
    ax.set_xlabel('Kills')
    ax.set_title(f'{phase_label}', fontsize=13)

plt.suptitle('Top Kill-Securing Heroes by Match Phase', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '06_heroes_by_phase')
plt.show()

---
## 6. Critical Hit Analysis

Critical hits (headshots) are the hallmark of mechanical skill in Overwatch. The `is_critical_hit` flag on kills tells us which final blows were headshots.

- Hitscan heroes (Widowmaker, Ashe, Cassidy, Sojourn) should have the highest crit rates
- Projectile heroes (Hanzo) can crit but less consistently
- Some heroes cannot headshot at all (Winston, Moira, Symmetra beam)

> **Coaching insight**: If your hitscan players have low crit kill rates compared to the dataset average, aim training should be a priority. If they're above average, you have a mechanical advantage to play around.

In [ ]:
# Critical hit rates by hero (for final blows that CAN crit)
crit_data = combat_kills.copy()
# is_critical_hit might be stored as 0/1 integers
crit_data['is_crit'] = crit_data['is_critical_hit'].astype(int)

hero_crit = crit_data.groupby('attacker_hero').agg(
    total_kills=('is_crit', 'count'),
    crit_kills=('is_crit', 'sum')
).reset_index()
hero_crit['crit_rate'] = hero_crit['crit_kills'] / hero_crit['total_kills']
hero_crit['role'] = hero_crit['attacker_hero'].map(HERO_ROLES)

# Filter to heroes with at least 200 kills and any crits
hero_crit_filtered = hero_crit[(hero_crit['total_kills'] >= 200) & (hero_crit['crit_kills'] > 0)]
hero_crit_filtered = hero_crit_filtered.sort_values('crit_rate', ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))
colors = [ROLE_COLORS.get(r, OW_COLORS['light_gray']) for r in hero_crit_filtered['role']]
bars = ax.barh(hero_crit_filtered['attacker_hero'], hero_crit_filtered['crit_rate'] * 100,
               color=colors, alpha=0.9)

for bar, (_, row) in zip(bars, hero_crit_filtered.iterrows()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{row["crit_rate"]*100:.1f}% ({row["crit_kills"]:.0f}/{row["total_kills"]:.0f})',
            va='center', fontsize=8, color=OW_COLORS['white'])

ax.set_xlabel('Critical Hit Kill Rate (%)')
ax.set_title('Critical Hit Rate on Final Blows by Hero (min 200 kills)', fontsize=14, fontweight='bold')

legend_elements = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '06_crit_rate_by_hero')
plt.show()

In [ ]:
# Overall critical hit kill stats
total_crits = crit_data['is_crit'].sum()
total_kills_counted = len(crit_data)
overall_crit_rate = total_crits / total_kills_counted * 100

print(f"Overall critical hit final blow rate: {overall_crit_rate:.1f}%")
print(f"Total crit kills: {total_crits:,} / {total_kills_counted:,}")
print()

# Crit rate by role
role_crit = crit_data.groupby('attacker_role').agg(
    total=('is_crit', 'count'),
    crits=('is_crit', 'sum')
)
role_crit['rate'] = role_crit['crits'] / role_crit['total'] * 100
print("Critical hit kill rate by role:")
for role in ['Tank', 'DPS', 'Support']:
    if role in role_crit.index:
        r = role_crit.loc[role]
        print(f"  {role}: {r['rate']:.1f}% ({r['crits']:.0f} / {r['total']:.0f})")

---
## 7. Summary & Coaching Implications

### Key Findings

| Finding | Detail |
|---------|--------|
| Primary Fire dominance | Most kills come from basic attacks, not abilities — fundamentals matter |
| Assist-win correlation | Teams with more assists (both types) tend to win — teamwork is measurable |
| FB ratio variation | DPS heroes vary widely in finish rate — some are chip damage dealers, others are closers |
| Kill timing | Kill rates shift across match phases, reflecting ult economy and fight cadence |
| Crit rates | Hitscan heroes predictably lead in crits, but the rates reveal mechanical skill ceilings |

### What Coaches Should Do

1. **Track ability kill breakdowns** for your team's DPS players — if they're relying too much on cooldowns and not enough on primary fire, it suggests inconsistent aim
2. **Monitor assist ratios** across matches — a team whose assist counts drop over time may be losing coordination
3. **Use final blow ratio** as a diagnostic for whether your DPS players are finishing targets or just poking
4. **Compare kill timing** to your team's tendencies — do you peak early or late? Match your comp to your strengths
5. **Benchmark crit rates** against this dataset to identify mechanical skill gaps